# EVORA-AI: ex ante analytical evaluation of the SAIA record

DSR cycle 1, artificial and formative evaluation in the FEDS sense (Venable et al., 2016). The notebook:

1. validates the illustrative RAG SAIA record against the EVORA-AI traceability invariant;
2. computes analytical indicators (coverage, automation potential, human-judgement load);
3. applies a material change (an external LLM update) and measures the scope of the selective revalidation.

**Scope.** The RAG system is hypothetical. The indicators describe the *design* of the record, not field effort, decision latency or effectiveness, which are left to the cycle 2 evaluation (F0–F4).

In [1]:
import sys, json
from datetime import date
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))
from evora import SAIARecord, ChangeEvent, validate, apply_change, indicators

EX = Path('..') / 'examples' / 'rag-citizen-service'
AS_OF = date(2026, 10, 29)
record = SAIARecord.model_validate_json((EX / 'saia.json').read_text())
event = ChangeEvent.model_validate_json((EX / 'change_llm_update.json').read_text())

## 1. Validation of the accepted record

In [2]:
def show(report):
    for f in report.findings:
        print(f'{f.severity.value:7} {f.rule:26} {f.ref:12} {f.message}')
    print(f'-> {len(report.errors)} error(s), {len(report.warnings)} warning(s); valid = {report.ok}')

show(validate(record, AS_OF))

warning R5-sufficiency             C-05         claim accepted under conditional sufficiency
-> 0 error(s), 1 warning(s); valid = True


## 2. Traceability chains (requirement → claim → control → evidence → role)

In [3]:
idx = record.index()
for c in record.claims:
    ev = record.evidence_for_claim(c)
    hooks = sum(1 for e in ev if e.automation_hook)
    print(f"{c.id} [{', '.join(p.value for p in c.phases)}] reqs={c.requirement_ids} controls={c.control_ids} "
          f"evidence={[e.id for e in ev]} hooked={hooks}/{len(ev)} accountable={c.accountable_role} reviewer={c.reviewer_role}")

C-01 [L3, L5] reqs=['REQ-LLM01', 'REQ-GUIA-GATE'] controls=['CTL-01'] evidence=['EV-01', 'EV-02'] hooked=1/2 accountable=product-owner reviewer=assurance-unit
C-02 [L2, L5] reqs=['REQ-LLM02', 'REQ-LGPD46', 'REQ-GUIA-GATE'] controls=['CTL-02', 'CTL-03'] evidence=['EV-03', 'EV-04', 'EV-05'] hooked=1/3 accountable=dpo reviewer=assurance-unit
C-03 [L2] reqs=['REQ-LLM04'] controls=['CTL-04'] evidence=['EV-06', 'EV-07'] hooked=2/2 accountable=data-steward reviewer=assurance-unit
C-04 [L4, L5] reqs=['REQ-LLM09', 'REQ-GUIA-GATE'] controls=['CTL-05'] evidence=['EV-08', 'EV-09'] hooked=1/2 accountable=product-owner reviewer=assurance-unit
C-05 [L3, L6] reqs=['REQ-LLM03'] controls=['CTL-06'] evidence=['EV-10', 'EV-11'] hooked=1/2 accountable=contract-mgr reviewer=assurance-unit


## 3. Selective revalidation after an external LLM update

In [4]:
updated = apply_change(record, event)
entry = updated.revalidation_history[-1]
print('change     :', event.component, entry.previous_version, '->', event.new_version)
print('reopened   :', entry.reopened_claim_ids)
print('unaffected :', entry.unaffected_claim_ids)
show(validate(updated, AS_OF))

change     : model:llm vendor-llm-2026-06 -> vendor-llm-2026-10
reopened   : ['C-01', 'C-04', 'C-05']
unaffected : ['C-02', 'C-03']
error   R5-sufficiency             C-01         claim reopened by a change; revalidation pending
error   R6-stale-evidence          C-01/EV-01   evidence bound to model:llm@vendor-llm-2026-06, deployed is vendor-llm-2026-10
error   R5-sufficiency             C-04         claim reopened by a change; revalidation pending
error   R6-stale-evidence          C-04/EV-08   evidence bound to model:llm@vendor-llm-2026-06, deployed is vendor-llm-2026-10
error   R5-sufficiency             C-05         claim reopened by a change; revalidation pending
error   R9-decision                decision     'proceed' with non-accepted claims: C-01, C-04, C-05
-> 6 error(s), 0 warning(s); valid = False


## 4. Analytical indicators

In [5]:
before, after = indicators(record), indicators(updated)
rows = ['requirements', 'requirement_coverage', 'claims', 'controls', 'evidence_items',
        'evidence_with_automation_hook', 'automation_share', 'human_judgement_fields',
        'reopened_claims_last_change', 'revalidation_scope_last_change']
print(f"{'indicator':32} value")
for k in rows:
    v = after.get(k, before.get(k))
    print(f"{k:32} {v:.2f}" if isinstance(v, float) else f"{k:32} {v}")

indicator                        value
requirements                     7
requirement_coverage             1.00
claims                           5
controls                         6
evidence_items                   11
evidence_with_automation_hook    6
automation_share                 0.55
human_judgement_fields           16
reopened_claims_last_change      3
revalidation_scope_last_change   0.60


**Reading.** All requirements are covered by at least one claim. 6 of the 11 evidence items can be harvested from existing engineering systems (CI, data catalogue, evaluation platform, model registry). Human effort concentrates on 16 judgement fields: statement, acceptance criterion and sufficiency for each claim, plus the decision. The model update reopens 3 of the 5 claims, and the 2 claims that do not depend on the model stay accepted.

These figures depend on how the illustrative record was modelled. They do not generalise without the cycle 2 evaluation.